In [2]:
# ============================================================
# Step 3
# Merge chunk TSVs into short_rawdata...tsv and long_rawdata...tsv
# Make sure that each bin is sorted and order matches
# ============================================================
import os
import glob
import pandas as pd
import numpy as np
import py2bit
import statsmodels.api as sm

INDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb"
OUTDIR = INDIR

os.makedirs(OUTDIR, exist_ok=True)

REMOVE_RAW_FILES = True


def chrom_rank(chrom):
    c = str(chrom).replace("chr", "").replace("CHR", "")

    if c.isdigit():
        return int(c)
    elif c == "X":
        return 23
    elif c == "Y":
        return 24
    elif c in ["M", "MT"]:
        return 25
    else:
        return 1000


def read_raw_tsv(path):
    df = pd.read_csv(path, sep="\t")

    # Force first 3 columns to standard names
    cols = list(df.columns)

    df = df.rename(
        columns={
            cols[0]: "chrom",
            cols[1]: "start",
            cols[2]: "end"
        }
    )

    df["chrom"] = df["chrom"].astype(str)
    df["start"] = pd.to_numeric(df["start"], errors="raise").astype(int)
    df["end"] = pd.to_numeric(df["end"], errors="raise").astype(int)

    return df


def get_raw_files(mode):
    return sorted(
        glob.glob(os.path.join(INDIR, f"{mode}.bamchunk_*.raw.tsv"))
    )


def get_sorted_files(mode):
    return sorted(
        glob.glob(os.path.join(INDIR, f"{mode}.bamchunk_*.sorted.tsv"))
    )


def ensure_sorted_chunks(mode):
    """
    If sorted chunks already exist, reuse them.
    If sorted chunks do not exist but raw chunks exist, sort raw chunks.
    """

    raw_files = get_raw_files(mode)
    sorted_files = get_sorted_files(mode)

    # Case 1: sorted files already exist and raw files are gone
    if len(sorted_files) > 0 and len(raw_files) == 0:
        print(
            f"\n[{mode}] Found {len(sorted_files)} existing sorted TSV files. "
            "Raw files are absent, so skipping sorting."
        )
        return sorted_files

    # Case 2: both raw and sorted files exist
    # Sort only missing sorted files, if any.
    if len(raw_files) > 0 and len(sorted_files) > 0:
        print(f"\n[{mode}] Raw files found: {len(raw_files)}")
        print(f"[{mode}] Existing sorted files found: {len(sorted_files)}")

        expected_sorted = {
            f.replace(".raw.tsv", ".sorted.tsv") for f in raw_files
        }

        existing_sorted = set(sorted_files)

        missing_sorted = sorted(expected_sorted - existing_sorted)

        if len(missing_sorted) == 0:
            print(f"[{mode}] All sorted files already exist. Skipping sorting.")
            return sorted_files

        print(f"[{mode}] Missing sorted files: {len(missing_sorted)}")
        print(f"[{mode}] Sorting only missing files.")

        raw_lookup = {
            f.replace(".raw.tsv", ".sorted.tsv"): f for f in raw_files
        }

        for sorted_path in missing_sorted:
            raw_path = raw_lookup[sorted_path]
            sort_one_raw_file(raw_path, sorted_path)

        sorted_files = get_sorted_files(mode)

        print(f"[{mode}] Sorted files now available: {len(sorted_files)}")
        return sorted_files

    # Case 3: no sorted files, but raw files exist
    if len(sorted_files) == 0 and len(raw_files) > 0:
        print(f"\n[{mode}] No sorted files found.")
        print(f"[{mode}] Sorting {len(raw_files)} raw files.")

        for raw_path in raw_files:
            sorted_path = raw_path.replace(".raw.tsv", ".sorted.tsv")
            sort_one_raw_file(raw_path, sorted_path)

        sorted_files = get_sorted_files(mode)

        print(f"[{mode}] Sorted files created: {len(sorted_files)}")
        return sorted_files

    # Case 4: neither raw nor sorted files exist
    raise FileNotFoundError(
        f"[{mode}] No raw or sorted chunk files found in {INDIR}. "
        f"Expected either {mode}.bamchunk_*.raw.tsv or "
        f"{mode}.bamchunk_*.sorted.tsv."
    )


def sort_one_raw_file(raw_path, sorted_path):
    df = read_raw_tsv(raw_path)

    df["__chrom_rank"] = df["chrom"].map(chrom_rank)

    df = df.sort_values(
        by=["__chrom_rank", "chrom", "start", "end"],
        kind="mergesort"
    ).drop(columns=["__chrom_rank"])

    # Write to temporary file first, then rename.
    # This prevents half-written sorted files if the kernel disconnects.
    tmp_path = sorted_path + ".tmp"

    df.to_csv(tmp_path, sep="\t", index=False)
    os.replace(tmp_path, sorted_path)

    print(
        f"  sorted: {os.path.basename(raw_path)} -> "
        f"{os.path.basename(sorted_path)}"
    )


def verify_sorted_bins_match(mode):
    files = get_sorted_files(mode)

    if not files:
        raise FileNotFoundError(f"No sorted TSV files found for mode: {mode}")

    print(f"\n[{mode}] Verifying sorted bin order across chunks")
    print(f"[{mode}] Sorted files found: {len(files)}")

    ref = pd.read_csv(
        files[0],
        sep="\t",
        usecols=["chrom", "start", "end"],
        dtype={"chrom": str}
    )

    ref["start"] = pd.to_numeric(ref["start"], errors="raise").astype(int)
    ref["end"] = pd.to_numeric(ref["end"], errors="raise").astype(int)

    ref_name = os.path.basename(files[0])

    print(f"  reference: {ref_name}")
    print(f"  reference bins: {ref.shape[0]}")

    for f in files[1:]:
        test = pd.read_csv(
            f,
            sep="\t",
            usecols=["chrom", "start", "end"],
            dtype={"chrom": str}
        )

        test["start"] = pd.to_numeric(test["start"], errors="raise").astype(int)
        test["end"] = pd.to_numeric(test["end"], errors="raise").astype(int)

        if not test.equals(ref):
            print("\nERROR: Bin coordinates/order do not match.")
            print(f"Reference file: {ref_name}")
            print(f"Problem file:   {os.path.basename(f)}")
            print(f"Reference rows: {ref.shape[0]}")
            print(f"Problem rows:   {test.shape[0]}")

            ref_diag = os.path.join(OUTDIR, f"{mode}_reference_bins.tsv")
            bad_diag = os.path.join(OUTDIR, f"{mode}_problem_bins.tsv")

            ref.to_csv(ref_diag, sep="\t", index=False)
            test.to_csv(bad_diag, sep="\t", index=False)

            raise ValueError(
                f"[{mode}] Sorted bins do not match across chunks. "
                "Do not merge until this is fixed."
            )

        print(f"  OK: {os.path.basename(f)}")

    print(f"[{mode}] All sorted chunks have identical bin order.")

    return True


def merge_sorted_chunks(mode):
    files = get_sorted_files(mode)

    if not files:
        raise FileNotFoundError(f"No sorted TSV files found for mode: {mode}")

    print(f"\n[{mode}] Merging {len(files)} sorted chunks")

    out = pd.read_csv(files[0], sep="\t", dtype={"chrom": str})

    out["start"] = pd.to_numeric(out["start"], errors="raise").astype(int)
    out["end"] = pd.to_numeric(out["end"], errors="raise").astype(int)

    coord = out[["chrom", "start", "end"]].copy()

    seen_samples = set(out.columns[3:])

    print(
        f"[{mode}] Starting with {len(seen_samples)} samples from "
        f"{os.path.basename(files[0])}"
    )

    for f in files[1:]:
        df = pd.read_csv(f, sep="\t", dtype={"chrom": str})

        df["start"] = pd.to_numeric(df["start"], errors="raise").astype(int)
        df["end"] = pd.to_numeric(df["end"], errors="raise").astype(int)

        if not df[["chrom", "start", "end"]].equals(coord):
            raise ValueError(
                f"[{mode}] Coordinate mismatch during merge: {os.path.basename(f)}"
            )

        sample_cols = list(df.columns[3:])

        dup = [s for s in sample_cols if s in seen_samples]

        if dup:
            raise ValueError(
                f"[{mode}] Duplicate sample names found in "
                f"{os.path.basename(f)}: {dup[:10]}"
            )

        seen_samples.update(sample_cols)

        out = pd.concat([out, df[sample_cols]], axis=1)

        print(
            f"  added {len(sample_cols)} samples from "
            f"{os.path.basename(f)}; total samples = {len(seen_samples)}"
        )

    out_path = os.path.join(OUTDIR, f"{mode}_rawdata_countreadsperbin.tsv")
    tmp_path = out_path + ".tmp"

    # Write to temp first to avoid incomplete final file if kernel disconnects
    out.to_csv(tmp_path, sep="\t", index=False)
    os.replace(tmp_path, out_path)

    print(f"\n[{mode}] Wrote: {out_path}")
    print(f"[{mode}] Final shape: {out.shape}")
    print(f"[{mode}] Final sample count: {len(seen_samples)}")

    return out_path


# ============================================================
# Ensure sorted chunks exist
# ============================================================

ensure_sorted_chunks("short")
ensure_sorted_chunks("long")


# ============================================================
# Verify sorted bins/order match before merging
# ============================================================

verify_sorted_bins_match("short")
verify_sorted_bins_match("long")


# ============================================================
# Merge only after successful verification
# ============================================================

short_path = merge_sorted_chunks("short")
long_path = merge_sorted_chunks("long")


# ============================================================
# Final short-vs-long bin check
# ============================================================

short_bins = pd.read_csv(
    short_path,
    sep="\t",
    usecols=["chrom", "start", "end"],
    dtype={"chrom": str}
)

long_bins = pd.read_csv(
    long_path,
    sep="\t",
    usecols=["chrom", "start", "end"],
    dtype={"chrom": str}
)

short_bins["start"] = pd.to_numeric(short_bins["start"], errors="raise").astype(int)
short_bins["end"] = pd.to_numeric(short_bins["end"], errors="raise").astype(int)

long_bins["start"] = pd.to_numeric(long_bins["start"], errors="raise").astype(int)
long_bins["end"] = pd.to_numeric(long_bins["end"], errors="raise").astype(int)

print("\nFinal short-vs-long bin check")

if not short_bins.equals(long_bins):
    raise ValueError(
        "Final merged short and long files do not have identical bins/order."
    )

print("Merged short/long bins identical: TRUE")
print("\nDone successfully.")


# ============================================================
# Remove raw TSV files after successful merge
# ============================================================

if REMOVE_RAW_FILES:
    raw_files = sorted(glob.glob(os.path.join(INDIR, "*.bamchunk_*.raw.tsv")))

    if len(raw_files) == 0:
        print("\nNo raw chunk files found to remove.")
    else:
        print("\nRemoving raw chunk files after successful merge:")

        for f in raw_files:
            os.remove(f)
            print(f"  removed: {os.path.basename(f)}")

        print(f"\nRemoved {len(raw_files)} raw TSV files.")


[short] Found 32 existing sorted TSV files. Raw files are absent, so skipping sorting.

[long] Found 32 existing sorted TSV files. Raw files are absent, so skipping sorting.

[short] Verifying sorted bin order across chunks
[short] Sorted files found: 32
  reference: short.bamchunk_00.sorted.tsv
  reference bins: 30894
  OK: short.bamchunk_01.sorted.tsv
  OK: short.bamchunk_02.sorted.tsv
  OK: short.bamchunk_03.sorted.tsv
  OK: short.bamchunk_04.sorted.tsv
  OK: short.bamchunk_05.sorted.tsv
  OK: short.bamchunk_06.sorted.tsv
  OK: short.bamchunk_07.sorted.tsv
  OK: short.bamchunk_08.sorted.tsv
  OK: short.bamchunk_09.sorted.tsv
  OK: short.bamchunk_10.sorted.tsv
  OK: short.bamchunk_11.sorted.tsv
  OK: short.bamchunk_12.sorted.tsv
  OK: short.bamchunk_13.sorted.tsv
  OK: short.bamchunk_14.sorted.tsv
  OK: short.bamchunk_15.sorted.tsv
  OK: short.bamchunk_16.sorted.tsv
  OK: short.bamchunk_17.sorted.tsv
  OK: short.bamchunk_18.sorted.tsv
  OK: short.bamchunk_19.sorted.tsv
  OK: short.b

In [3]:
# Step 4 — Blacklist filtering, GC content, and LOESS GC correction

import os
import glob
import pandas as pd
import bisect

SHORT_IN = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_rawdata_countreadsperbin.tsv"
LONG_IN = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/long_rawdata_countreadsperbin.tsv"
BLACKLIST = "/ix1/alee/LO_LAB/Personal/Rahul/Reference_genome/Human/Blacklist_file/hg38-blacklist.v2.bed"

SHORT_OUT = SHORT_IN.replace(".tsv", ".blacklistFiltered.tsv")
LONG_OUT  = LONG_IN.replace(".tsv",  ".blacklistFiltered.tsv")

# -------------------------
# Load blacklist into per-chrom interval lists
# -------------------------
bl = pd.read_csv(BLACKLIST, sep="\t", header=None, comment="#", usecols=[0,1,2])
bl.columns = ["chrom","start","end"]

# keep only UCSC main chroms if you want (recommended)
keep = {f"chr{i}" for i in range(1,23)} | {"chrX"}   # add "chrY" if needed
bl = bl[bl["chrom"].isin(keep)].copy()

# sort and store starts/ends as python lists per chrom
bl_dict = {}
for chrom, sub in bl.groupby("chrom"):
    sub = sub.sort_values("start")
    bl_dict[chrom] = (sub["start"].astype(int).tolist(), sub["end"].astype(int).tolist())

def overlaps_blacklist(chrom, start, end):
    """Return True if [start,end) overlaps any blacklist interval on chrom."""
    if chrom not in bl_dict:
        return False
    starts, ends = bl_dict[chrom]
    # find rightmost interval with start < end
    i = bisect.bisect_left(starts, end) - 1
    if i >= 0 and ends[i] > start:
        return True
    # also check the next interval (in case bisect landed before overlap)
    j = i + 1
    if 0 <= j < len(starts) and starts[j] < end and ends[j] > start:
        return True
    return False

# -------------------------
# Filter a merged CRPB matrix
# -------------------------
def filter_matrix(in_path, out_path):
    df = pd.read_csv(in_path, sep="\t", dtype={"chrom": str})
    # optional: restrict to chr1-22,X here too (keeps things clean)
    df = df[df["chrom"].isin(keep)].copy()

    mask = df.apply(lambda r: overlaps_blacklist(r["chrom"], int(r["start"]), int(r["end"])), axis=1)
    n_bad = int(mask.sum())
    print(f"{in_path}: bins={df.shape[0]}, blacklisted={n_bad}, kept={df.shape[0]-n_bad}")

    df_f = df[~mask].copy()
    df_f.to_csv(out_path, sep="\t", index=False)
    return out_path

short_f = filter_matrix(SHORT_IN, SHORT_OUT)
long_f  = filter_matrix(LONG_IN,  LONG_OUT)

# sanity: short/long bins identical after filtering
s = pd.read_csv(short_f, sep="\t", usecols=["chrom","start","end"])
l = pd.read_csv(long_f,  sep="\t", usecols=["chrom","start","end"])
print("After blacklist filter, short/long bins identical:", s.equals(l))
print("Wrote:", short_f)
print("Wrote:", long_f)


/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_rawdata_countreadsperbin.tsv: bins=30321, blacklisted=2529, kept=27792
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/long_rawdata_countreadsperbin.tsv: bins=30321, blacklisted=2529, kept=27792
After blacklist filter, short/long bins identical: True
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_rawdata_countreadsperbin.blacklistFiltered.tsv
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/long_rawdata_countreadsperbin.blacklistFiltered.tsv


In [ ]:
# Step 5 — Calculate GC content for each filtered bin + LOESS correction
# Take more time than expected - decided to run from terminal

INDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb"

SHORT_IN = os.path.join(
    INDIR,
    "short_rawdata_countreadsperbin.blacklistFiltered.tsv"
)

LONG_IN = os.path.join(
    INDIR,
    "long_rawdata_countreadsperbin.blacklistFiltered.tsv"
)

TWOBIT = "/ix1/alee/LO_LAB/Personal/Rahul/Reference_genome/Human/Human_ref_hg38_p7/hg38.2bit"

OUTDIR = INDIR

# Load genome
tb = py2bit.open(TWOBIT)


def gc_fraction_from_2bit(tb, chrom, start, end):
    """
    Calculate GC fraction for one genomic interval using py2bit.
    Handles chr/non-chr chromosome naming differences.
    """

    chrom = str(chrom)
    start = int(start)
    end = int(end)

    chroms = tb.chroms()

    # Try chromosome as-is first
    chrom_use = chrom

    # If not present, try chr conversion
    if chrom_use not in chroms:
        if chrom.startswith("chr"):
            alt_chrom = chrom.replace("chr", "", 1)
        else:
            alt_chrom = "chr" + chrom

        if alt_chrom in chroms:
            chrom_use = alt_chrom
        else:
            return np.nan

    chrom_size = chroms[chrom_use]

    # Protect against coordinates outside chromosome length
    start = max(0, start)
    end = min(end, chrom_size)

    if end <= start:
        return np.nan

    try:
        seq = tb.sequence(chrom_use, start, end)
    except Exception:
        return np.nan

    if seq is None:
        return np.nan

    seq = seq.upper()

    valid = (
        seq.count("A") +
        seq.count("C") +
        seq.count("G") +
        seq.count("T")
    )

    if valid == 0:
        return np.nan

    gc_count = seq.count("G") + seq.count("C")

    return gc_count / valid


# Quick checks
df_test = pd.read_csv(SHORT_IN, sep="\t", nrows=5)
print(df_test[["chrom", "start", "end"]])

chrom_keys = list(tb.chroms().keys())[:20]
print("First 20 chroms in 2bit:", chrom_keys)
print("chr1 in 2bit?", "chr1" in tb.chroms())
print("1 in 2bit?", "1" in tb.chroms())


def gc_loess_correct(in_tsv, out_prefix, frac=0.75):
    print("\nProcessing:", in_tsv)

    df = pd.read_csv(in_tsv, sep="\t")

    # Make sure coordinate columns are standard
    df = df.rename(
        columns={
            df.columns[0]: "chrom",
            df.columns[1]: "start",
            df.columns[2]: "end"
        }
    )

    df["start"] = pd.to_numeric(df["start"], errors="raise").astype(int)
    df["end"] = pd.to_numeric(df["end"], errors="raise").astype(int)

    samplenames = list(df.columns[3:])

    print("  bins:", df.shape[0])
    print("  samples:", len(samplenames))

    # ---- GC content ----
    gc = [
        gc_fraction_from_2bit(tb, c, s, e)
        for c, s, e in zip(df["chrom"], df["start"], df["end"])
    ]

    df["GC"] = gc

    print("  GC NaNs:", int(df["GC"].isna().sum()), "out of", df.shape[0])

    df = df.dropna(subset=["GC"]).copy()

    print("  bins after GC filter:", df.shape[0])

    # Mean readcount across samples
    df["mean_readcount_over_all_samples"] = df[samplenames].mean(axis=1)

    # ---- LOESS per sample ----
    df = df.sort_values("GC").reset_index(drop=True)

    corrected = {}

    for i, sample in enumerate(samplenames, 1):
        y = df[sample].values
        x = df["GC"].values

        loess = sm.nonparametric.lowess(
            endog=y,
            exog=x,
            frac=frac,
            is_sorted=True,
            return_sorted=False
        )

        corrected[sample] = y - loess + np.median(y)

        if i % 25 == 0 or i == len(samplenames):
            print("    LOESS done:", i, "/", len(samplenames))

    # ---- Save outputs ----
    df_corr = pd.DataFrame(corrected)

    df_corr.insert(0, "end", df["end"].values)
    df_corr.insert(0, "start", df["start"].values)
    df_corr.insert(0, "chrom", df["chrom"].values)

    # Full table with GC
    out1 = out_prefix + "_readcounts_corr_and_GC.csv"

    df_out = df[[
        "chrom",
        "start",
        "end",
        "GC",
        "mean_readcount_over_all_samples"
    ]].join(df_corr[samplenames])

    df_out.to_csv(out1, index=False)

    # Matrix form: samples x bins
    df_corr["bin"] = (
        df["chrom"].astype(str) + "_" +
        df["start"].astype(str) + "_" +
        df["end"].astype(str)
    )

    mat = df_corr.set_index("bin")[samplenames].transpose()

    out2 = out_prefix + "_corr_readcounts.csv"
    mat.to_csv(out2)

    print("  wrote:")
    print("   ", out1)
    print("   ", out2)

    return mat


short_mat = gc_loess_correct(
    SHORT_IN,
    os.path.join(OUTDIR, "short")
)

long_mat = gc_loess_correct(
    LONG_IN,
    os.path.join(OUTDIR, "long")
)

print("short matrix:", short_mat.shape)
print("long matrix:", long_mat.shape)

assert short_mat.index.equals(long_mat.index)
assert short_mat.columns.equals(long_mat.columns)

print("Short/long matrices aligned")

tb.close()


  chrom    start      end
0  chr1   800000   900000
1  chr1   900000  1000000
2  chr1  1000000  1100000
3  chr1  1100000  1200000
4  chr1  1200000  1300000
First 20 chroms in 2bit: ['chr1', 'chr10', 'chr11', 'chr11_KI270721v1_random', 'chr12', 'chr13', 'chr14', 'chr14_GL000009v2_random', 'chr14_GL000225v1_random', 'chr14_KI270722v1_random', 'chr14_GL000194v1_random', 'chr14_KI270723v1_random', 'chr14_KI270724v1_random', 'chr14_KI270725v1_random', 'chr14_KI270726v1_random', 'chr15', 'chr15_KI270727v1_random', 'chr16', 'chr16_KI270728v1_random', 'chr17']
chr1 in 2bit? True
1 in 2bit? False

Processing: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_rawdata_countreadsperbin.blacklistFiltered.tsv
  bins: 27792
  samples: 784
  GC NaNs: 272 out of 27792
  bins after GC filter: 27520


KeyboardInterrupt: 

Exception ignored in: 'statsmodels.nonparametric._smoothers_lowess.calculate_weights'
Traceback (most recent call last):
  File "<__array_function__ internals>", line 2, in sum
KeyboardInterrupt


    LOESS done: 25 / 784


In [3]:
import pandas as pd

indir = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb"

long_full = pd.read_csv(f"{indir}/long_readcounts_corr_and_GC.csv")
long_mat  = pd.read_csv(f"{indir}/long_corr_readcounts.csv", index_col=0)

print("Full GC file shape:", long_full.shape)
print("Matrix file shape:", long_mat.shape)

display(long_full.head(2))
display(long_mat.head(2))

Full GC file shape: (27520, 789)
Matrix file shape: (784, 27520)


,chrom,start,end,GC,mean_readcount_over_all_samples,AL1Y-NB1,AL1Y-NB2,AL1Y-NB3,AL1Y-NB4,AL1Y-NB5,...,ARVO-NB1,ARVO-NB2,ARVO-NB3,ARVO-NB4,ARVV-NB1,ARVV-NB2,ARWN-NB1,ARWN-NB2,ARWN-NB3,ARWN-NB4
0,chr3,165000000,165100000,0.31244,132.613520,181.811169,216.297659,180.309879,166.072459,149.571752,...,276.590581,169.470499,109.960023,156.821032,214.085711,155.327701,140.650542,159.756410,155.121222,149.510732
1,chr8,112900000,113000000,0.32195,152.118622,165.285632,273.341211,195.900800,178.848628,173.119031,...,288.330179,191.153772,132.177382,161.563095,204.309141,156.798185,123.136202,143.211482,180.638176,153.423872


,chr3_165000000_165100000,chr8_112900000_113000000,chr13_55700000_55800000,chr2_193300000_193400000,chr6_102500000_102600000,chr18_66500000_66600000,chr13_71500000_71600000,chr2_193100000_193200000,chr6_95000000_95100000,chr6_67100000_67200000,...,chr1_3500000_3600000,chr16_700000_800000,chr16_500000_600000,chr1_1200000_1300000,chr1_2400000_2500000,chr1_1000000_1100000,chr16_1000000_1100000,chr16_88400000_88500000,chr19_1200000_1300000,chr16_1100000_1200000
AL1Y-NB1,181.811169,165.285632,190.504051,178.454184,166.406955,180.380724,154.370233,174.199854,94.027058,177.993047,...,198.866289,166.928078,152.978615,144.005791,174.013169,172.033939,178.033939,171.050113,121.330850,215.601354
AL1Y-NB2,216.297659,273.341211,198.428812,278.370643,229.315559,272.284966,254.272731,266.074067,113.872655,209.833020,...,281.651052,231.890252,167.085350,257.190065,236.218471,254.298381,296.298381,287.360553,160.432484,271.453435


In [1]:
# Step 6 — Compute short/long fragmentation ratio
import os
import glob
import pandas as pd
import numpy as np
import py2bit
import statsmodels.api as sm

INDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb"

SHORT = f"{INDIR}/short_corr_readcounts.csv"
LONG  = f"{INDIR}/long_corr_readcounts.csv"

# Load
short_df = pd.read_csv(SHORT, index_col=0)
long_df  = pd.read_csv(LONG,  index_col=0)

# Safety checks
assert short_df.shape == long_df.shape
assert (short_df.columns == long_df.columns).all()

print("Matrix shape:", short_df.shape)

# Avoid division by zero
eps = 1e-6
ratio_df = short_df / (long_df + eps)

# Save
ratio_out = f"{INDIR}/short_long_ratio.csv"
ratio_df.to_csv(ratio_out)

print("Wrote:", ratio_out)


Matrix shape: (784, 27520)
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_long_ratio.csv


In [2]:
import pandas as pd

file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Fragmentation_analysis/Regional_Fragment_analysis/02_crpb/short_long_ratio.csv"

df = pd.read_csv(file_path)
df.head(2)

,Unnamed: 0,chr3_165000000_165100000,chr8_112900000_113000000,chr13_55700000_55800000,chr2_193300000_193400000,chr6_102500000_102600000,chr18_66500000_66600000,chr13_71500000_71600000,chr2_193100000_193200000,chr6_95000000_95100000,...,chr1_3500000_3600000,chr16_700000_800000,chr16_500000_600000,chr1_1200000_1300000,chr1_2400000_2500000,chr1_1000000_1100000,chr16_1000000_1100000,chr16_88400000_88500000,chr19_1200000_1300000,chr16_1100000_1200000
0,AL1Y-NB1,0.293926,0.273176,0.268041,0.151620,0.210639,0.216480,0.110434,0.235524,0.265972,...,0.209562,0.177755,0.154730,0.219917,0.210726,0.114328,0.205962,0.126675,0.178501,0.146793
1,AL1Y-NB2,0.320351,0.215495,0.286162,0.189586,0.221389,0.164401,0.270427,0.213233,0.164297,...,0.137358,0.192768,0.279573,0.243861,0.206252,0.215200,0.157696,0.260052,0.391395,0.165234
